In [1]:
%pip install "pandas>=2.0.0" "openai>=2.20.0" "python-dotenv>=1.0.0"

     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     ---------------------------------------- 90.6/90.6 kB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------------------------------- ------ 1.0/1.1 MB 20.3 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 18.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/204.6 kB ? eta -:--:--
   --------------------------------------- 204.6/204.6 kB 12.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/463.6 kB ? eta -:--:--
   --------------------------------------- 463.6/463.6 kB 28.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------------------------- ------ 1.7/2.0 MB 35.9 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 31.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 합성 데이터

In [2]:
import os
import json
from pprint import pprint

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
# 1. load_dotenv() 함수를 사용하여 .env 파일을 로드하세요
load_dotenv()

# 2. os.getenv() 함수를 사용하여 UPSTAGE_API_KEY를 가져오세요
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

# 정상적으로 잘 작동하는지 확인해봅시다.
if UPSTAGE_API_KEY:
    print("Success API Key Setting!")
else:
    print("Failed to load API Key. Please check your .env file.")

Success API Key Setting!


In [4]:
# 1. OpenAI 클라이언트 생성
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# 2. chat_completion 함수를 완성
def chat_completion(prompt: str,
                    system_prompt: str = None,
                    model: str = "solar-pro3") -> str:
    """
    LLM을 호출하여 응답을 반환하는 함수

    Args:
        prompt: 사용자 메시지
        system_prompt: 시스템 메시지 (선택)
        model: 사용할 모델명

    Returns:
        LLM의 응답 텍스트
    """
    messages = []

    # 시스템 프롬프트는 LLM의 동작을 제어하는 프롬프트로
    # 대화의 스타일, 성격, 제약 조건 등을 결정합니다.
    # LLM이 어떤 방식으로 답변해야 하는지를 미리 정해주는 지침으로 이해하면 됩니다.
    # 필수는 아니기 때문에 `if`문으로 처리하였습니다.
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# 잘 작동하는지 테스트해볼까요?
test_response = chat_completion("안녕하세요! 간단한 인사말을 해주세요.")
print("테스트 응답:")
print(test_response)

테스트 응답:
안녕하세요! 만나서 반갑습니다. 😊  
좋은 하루 보내고 계신가요?


## Prompt Engineering

In [5]:
# 공통 시스템 프롬프트
SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다.
추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다."""

user_query = "스릴러 영화를 추천해줘"

# 1. Zero-shot 프롬프트 `zero_shot_prompt`를 작성하세요 (예시 없이 직접 지시)
zero_shot_prompt = f'{user_query}'

print("=" * 50)
print("Zero-shot 프롬프팅 결과:")
print("=" * 50)
zero_shot_result = chat_completion(zero_shot_prompt, SYSTEM_PROMPT)
print(zero_shot_result)
print()

Zero-shot 프롬프팅 결과:
**추천 스릴러 영화**  

| 영화 | 개봉 연도 | 추천 이유 |
|------|----------|----------|
| **《기생충》** | 2019 | 사회적 계층 갈등을 치밀하게 파고들며, 반전과 긴장감이 끊이지 않는 연출이 돋보입니다. 단순한 스릴러 이상으로 인간 본성을 탐구하는 깊이 있는 메시지를 담고 있어, 보는 내내 마음을 놓지 못합니다. |
| **《사이코》** | 1960 | 알프레드 히치코크의 고전. 심리적 긴장과 서스펜스가 점진적으로 고조되며, 충격적인 결말이 아직도 많은 감독들에게 영감을 줍니다. 흑백 화면과 절제된 사운드가 오히려 더 강렬한 공포를 만들어냅니다. |
| **《세븐》** | 1995 | ‘7가지 대죄’를 주제로 한 살인 사건을 추적하는 두 탐정의 어두운 여정이 뛰어난 연출과 음악과 어우러져, 처음부터 끝까지 빈틈없는 스릴을 제공합니다. |
| **《마더 뒤팽》** | 2018 | 한국 스릴러 중에서도 독특한 반전과 충격적인 전개가 인상적인 작품. 과거와 현재를 오가는 플래시백과 블랙 코미디 요소가 결합해, 보는 내내 놀라게 합니다. |
| **《노인을 위한 나라는 없다》** | 2007 | 코엔 형제의 작품으로, 무자비한 살인마와 운명의 흐름을 따라가며 인간의 무력함을 탐구합니다. 전형적인 스릴러와는 차별화된 철학적 깊이가 있어, 감상 후 여운이 오래 남습니다. |

위 영화들은 장르적 긴장감과 스토리텔링 모두 뛰어나니, 취향에 맞게 골라 보시길 권합니다! 😊



In [7]:
# 2. Few-shot 프롬프트 `few_shot_prompt`를 작성하세요 (2-3개의 예시 포함)
few_shot_prompt = f"""다음은 영화 추천 예시입니다:

질문: 경제 영화를 추천해줘
답변: 영화 제목: 빅쇼트 (The Big Short)
개봉 연도: 2015년
추천 이유: 제2차 세계 대전 이후 미국 최대, 최악의 금융 위기를 쉽게 이해할 수 있는 구성과 더불어, 뛰어난 편집, 연출이 돋보이는 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 총각파티 후 기억을 잃은 친구들의 황당한 모험을 그린 코미디로, 예측 불가능한 전개와 유쾌한 유머가 가득합니다.

질문: {user_query}
답변: """

print("=" * 50)
print("Few-shot 프롬프팅 결과:")
print("=" * 50)
few_shot_result = chat_completion(few_shot_prompt, SYSTEM_PROMPT)
pprint(few_shot_result)
print()

Few-shot 프롬프팅 결과:
('영화 제목: 세븐 (Se7en)  \n'
 '개봉 연도: 1995년  \n'
 '추천 이유: 도시를 배경으로 한 연쇄 살인마와 두 수사관의 치열한 두뇌 싸움을 통해 숨막히는 긴장감과 충격적인 반전을 선사하는 클래식 '
 '스릴러이다.')



In [8]:
# 3. CoT 프롬프트 `cot_prompt`를 작성하세요 (단계별 추론 유도)
cot_prompt = f"""{user_query}

영화를 추천하기 전에 다음 단계를 따라 생각해주세요:

1단계: 스릴러 장르의 핵심 요소가 무엇인지 정의합니다 (긴장감, 반전, 서스펜스 등)
2단계: 이 요소들을 잘 갖춘 대표적인 스릴러 영화들을 떠올립니다
3단계: 그 중에서 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요."""

print("=" * 50)
print("Chain-of-Thought 프롬프팅 결과:")
print("=" * 50)
cot_result = chat_completion(cot_prompt, SYSTEM_PROMPT)
print(cot_result)
print()

Chain-of-Thought 프롬프팅 결과:
### 1단계: 스릴러 장르의 핵심 요소 정의  
스릴러 장르는 **긴장감**, **서스펜스**, **반전**, **심리적 압박**, **시간의 압박** 등을 주된 요소로 합니다. 특히 관객이 예측하지 못하는 전개와 결말을 통해 끝까지 몰입하게 만드는 것이 핵심입니다.  

---

### 2단계: 핵심 요소를 잘 갖춘 대표 스릴러 영화 떠올리기  
- 《식스 센스》(1999): 반전의 교과서  
- 《메멘토》(2000): 기억 상실과 시간 구조의 반전  
- 《곤지암》(2020): 심리적 압박과 공포의 극대화  
- 《버닝》(2018): 서스펜스와 미스터리의 조화  
- 《올드보이》(2003): 충격적인 반전과 폭력적 긴장감  

---

### 3단계: 최종 추천 영화 선정 및 이유  

**📽 추천 영화**: 《식스 센스》(1999)  

**✅ 추천 이유**:  
- **완벽한 반전**: 영화 전체를 통해 쌓은 긴장감이 결말에서 완전히 뒤집히며, 관객이 "다시 보기"를 반드시 하게 만드는 구조입니다.  
- **서스펜스의 지속**: 초반부터 미묘한 심리적 긴장감을 유지하며, 끝까지 진실을 숨기는 연출이 뛰어납니다.  
- **심리의 깊이**: 주인공의 트라우마와 죽은 사람과의 대화가 주는 철학적 고민이 스릴러와 결합되어 지루함 없이 전개됩니다.  

이 영화는 스릴러 장르의 모든 요소를 완벽하게 보여주는 클래식 작품으로, 처음 보는 관객도 끝까지 숨죽이며 지켜보게 만드는 매력을 지녔습니다. 🎬



## 구조화된 합성 데이터 생성

In [9]:
def json_parsing(output_text: str) -> dict:
    """
    LLM 응답에서 JSON 부분을 추출하여 딕셔너리로 변환

    Args:
        output_text: LLM의 응답 텍스트

    Returns:
        파싱된 딕셔너리
    """
    try:
        # ```json ... ``` 형식에서 JSON 추출
        if "```json" in output_text:
            output_text = output_text[output_text.index("```json") + len("```json"):].strip()
            output_text = output_text[:output_text.index("```")].strip()
        return json.loads(output_text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"JSON 파싱 오류: {e}")
        return None

In [10]:
# 1. 구조화된 출력을 위한 시스템 프롬프트를 작성하세요
STRUCTURED_GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

## 1. 입력 형식
[추천받고자 하는 영화 장르]

## 2. 작업 지시
- 요청된 장르에 가장 적합한 영화 1개를 추천합니다.
- 추천 이유는 구체적이고 설득력 있게 작성합니다.
- 친근하고 유머러스한 말투로 설명합니다.

## 3. 출력 형식
- 출력 형식은 다음 포맷을 따릅니다.
```json
{
    "movie_name": [영화 이름],
    "year": [개봉 연도],
    "genre": [장르],
    "reason": [추천 이유]
}
```
"""

# 2. 1에서 만든 프롬프트를 활용하여 답변을 받고,
# 답변을 JSON으로 파싱하여 `dict` 형태로 반환하도록 함수를 구성해주세요.
def generate_movie_recommendation(genre: str, temperature: float = 1.0) -> dict:
    """
    특정 장르에 대한 영화 추천 데이터를 생성

    Args:
        genre: 영화 장르
        temperature: 다양성 조절 파라미터 (0.0~2.0)

    Returns:
        구조화된 영화 추천 데이터
    """
    response = client.chat.completions.create(
        model='solar-pro3',
        messages=[
            {'role': 'system', 'content': STRUCTURED_GENERATOR_SYSTEM_PROMPT},
            {'role': 'user', 'content': f'{genre} 영화를 추천해줘'}
        ]
    )
    output = response.choices[0].message.content
    return json_parsing(output)

# 3. 3가지 다른 장르에 대해 합성 데이터를 생성하고 `synthetic_data`에 저장해주세요.
genres = ["공포", "SF", "액션"]
synthetic_data = []

for genre in genres:
    print(f'\n{genre} 장르 데이터 생성 중...')
    data = generate_movie_recommendation(genre, temperature=1.0)
    if data:
        synthetic_data.append(data)
        print('생성 완료: ', end='')
        pprint(data)

# 생성된 데이터 확인
print("\n" + "=" * 50)
print("생성된 합성 데이터:")
print("=" * 50)
for i, data in enumerate(synthetic_data, 1):
    print(f"\n[{i}] ", end=""); pprint(data)


공포 장르 데이터 생성 중...
생성 완료: {'genre': '공포',
 'movie_name': '컨저링',
 'reason': '이 영화는 진짜 귀신이 등장하는 게 아니라, 귀신이 살 정도로 집이 끔찍한 모양이라 무서워요! 오래된 저택의 음산한 '
           "분위기와 실제로 일어난 듯한 생생한 귀신 이야기, 그리고 '컨저링' 시리즈 특유의 현실적인 공포감이 너무 좋아서, 공포 "
           '영화 초보자도 쉽게 빠져들게 돼요. 게다가 가족이 함께 봐도 되는 적당한 무서운 수준이라, 친구들끼리 모여서 보기 딱 '
           '좋답니다. 한번 보면 밤에 화장실 갈 때 조명이 꺼진 복도가 진짜 무섭게 느껴질 거예요. 😱',
 'year': 2013}

SF 장르 데이터 생성 중...
생성 완료: {'genre': 'SF',
 'movie_name': '인터스텔라',
 'reason': '시간 여행과 우주 탐험이라는 SF의 고전적인 매력을 압도적인 시각 효과와 감동적인 스토리로 풀어낸 작품이에요. 블랙홀, '
           '5차원 공간, 사랑이라는 과학적·수학적 개념을 친숙하게 풀어내면서도, 가족, 희생, 인류의 미래를 생각하게 만드는 깊은 '
           '메시지가 담겨 있답니다. 혹시 아직도 이 영화를 안 보셨다면, 꼭 한 번은 4DX 혹은 아이맥스로 체험해 보세요. 우주 '
           '여행이 눈앞으로 펼쳐지는 느낌을 직접 느낄 수 있을 거예요!',
 'year': 2014}

액션 장르 데이터 생성 중...
생성 완료: {'genre': '액션',
 'movie_name': '매드맥스: 분노의 도로',
 'reason': '폭주하는 포스트아포칼립스 세계에서 미친 듯이 달려라! 미친 듯한 차량 추격과 폭발, 그리고 샤를리즈 테론의 강철 같은 '
           '카리스마가 한 번에 터진다. 액션은 기본, 스토리도 재미있고, 눈도 즐거운 이 영화를 보면 심장이 터질 듯이 뛰는 걸 '
          

In [11]:
synthetic_data[0]['movie_name']

'컨저링'